# 面试问题：LLM 输出 PII 门禁怎样做检测、脱敏和阻断？

可以直接复述的回答是：第一，输出离开模型前必须进入独立策略层。第二，检测器同时处理邮箱、电话、证件、银行卡和地址，并先做 Unicode 规范化。第三，联系方式可按授权场景掩码，高敏证件和银行卡默认阻断整条输出。第四，结构化 JSON 的每个字段都要递归扫描。第五，检测结果应保存类型和规则版本，不保存明文秘密。第六，用逐样本泄漏率、误拦截和绕过样本验证。下面使用五条完全虚构的客服输出演示。

## 真实案例：客服 Agent 回复中的虚构个人信息

五条教学输出分别包含带分隔符电话和 `.invalid` 邮箱、虚构证件号、虚构银行卡号、示例地址以及无 PII 订单状态。号码只用于规则测试，不属于真实用户。目标是决定 mask、block 或 allow，并输出策略证据。

In [1]:
outputs = [  # 定义五条具有不同 PII 风险的虚构客服回复
    {"id": "PII-01", "text": "联系人电话 138-0000-0000，邮箱 demo.user@example.invalid。", "expected": "mask"},  # 联系方式可掩码后返回
    {"id": "PII-02", "text": "系统记录的虚构身份证号为 110000200001010019。", "expected": "block"},  # 身份证属于高敏数据
    {"id": "PII-03", "text": "退款将进入虚构卡号 6222 0200 0000 0000。", "expected": "block"},  # 银行卡属于金融秘密
    {"id": "PII-04", "text": "配送到示例市测试路 88 号，请确认。", "expected": "mask"},  # 精确地址需要掩码
    {"id": "PII-05", "text": "订单 A-100 已退款，预计三个工作日到账。", "expected": "allow"},  # 不含个人信息的普通回复
]  # 结束五条策略评测输入
print("模型输出输入：id | expected | text")  # 展示策略层实际接收的文本
for item in outputs:  # 逐条输出五个虚构样本
    print(f"{item['id']} | {item['expected']:5} | {item['text']}")  # 呈现电话、证件、卡号、地址和安全文本
print("声明：以上号码、地址和邮箱均为教学虚构数据")  # 明确保存输出不含真实 PII


模型输出输入：id | expected | text
PII-01 | mask  | 联系人电话 138-0000-0000，邮箱 demo.user@example.invalid。
PII-02 | block | 系统记录的虚构身份证号为 110000200001010019。
PII-03 | block | 退款将进入虚构卡号 6222 0200 0000 0000。
PII-04 | mask  | 配送到示例市测试路 88 号，请确认。
PII-05 | allow | 订单 A-100 已退款，预计三个工作日到账。
声明：以上号码、地址和邮箱均为教学虚构数据


## Baseline / 基线：只匹配连续电话与普通邮箱

简单正则可以找到邮箱，却漏掉带横线电话、空格银行卡、证件和地址。逐样本输出显示漏检类型。

In [2]:
import re  # 使用正则实现基础和核心 PII 检测
import unicodedata  # 使用 NFKC 处理全角数字等 Unicode 绕过
baseline_email = re.compile(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}")  # 定义普通邮箱模式
baseline_phone = re.compile(r"1[3-9]\d{9}")  # 定义只接受连续十一位数字的脆弱电话模式
def baseline_detect(text):  # 返回基线可识别的 PII 类型
    types = []  # 收集邮箱和连续电话命中
    if baseline_email.search(text):  # 检查普通邮箱格式
        types.append("email")  # 记录邮箱类型
    if baseline_phone.search(text):  # 检查无分隔符手机号码
        types.append("phone")  # 记录电话类型
    return types  # 返回有限检测结果
print("基线检测：id | detected_types | leaked_high_risk")  # 输出五条样本的漏检情况
for item in outputs:  # 逐样本运行简单规则
    detected = baseline_detect(item["text"])  # 获取基线命中类型
    leaked = item["expected"] == "block" and not detected  # 高敏样本未命中即构成泄漏
    print(f"{item['id']} | {detected} | {leaked}")  # 展示分隔符和未知类型导致的失败


基线检测：id | detected_types | leaked_high_risk
PII-01 | ['email'] | False
PII-02 | [] | True
PII-03 | [] | True
PII-04 | [] | False
PII-05 | [] | False


## 核心实现：Unicode 规范化、多类型 Span 与策略动作

电话和银行卡模式允许空格或横线，地址只匹配本教学中的“市…路…号”结构。检测器只把类型和掩码文本交给日志，不输出高敏原值。

In [3]:
patterns = {  # 定义五类 PII 的可审计检测规则
    "email": re.compile(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}"),  # 检测邮箱地址
    "phone": re.compile(r"1[3-9](?:[ -]?\d){9}"),  # 检测带空格或横线的十一位手机号码
    "national_id": re.compile(r"(?<!\d)\d{17}[0-9Xx](?!\d)"),  # 检测十八位证件格式
    "bank_card": re.compile(r"(?<!\d)(?:\d[ -]?){15,18}\d(?!\d)"),  # 检测十六至十九位银行卡格式
    "address": re.compile(r"[一-鿿]{2,8}市[一-鿿]{2,12}路\s*\d+\s*号"),  # 检测示例城市道路门牌结构
}  # 结束多类型规则集合
high_risk_types = {"national_id", "bank_card"}  # 定义命中后阻断整条输出的高敏类型
def normalize_text(text):  # 统一全角字符并移除零宽字符
    normalized = unicodedata.normalize("NFKC", text)  # 把全角数字转换为 ASCII 形式
    return normalized.replace("​", "")  # 移除常见零宽空格绕过
def inspect_output(text):  # 检测、掩码并给出策略动作
    normalized = normalize_text(text)  # 在所有规则前执行同一规范化
    findings = []  # 收集类型和字符跨度而不持久化原值
    for pii_type, pattern in patterns.items():  # 逐类型扫描规范化输出
        for match in pattern.finditer(normalized):  # 获取当前类型的所有匹配 Span
            findings.append({"type": pii_type, "start": match.start(), "end": match.end()})  # 只记录类型和位置
    detected_types = {finding["type"] for finding in findings}  # 汇总当前输出包含的 PII 类型
    action = "block" if detected_types & high_risk_types else ("mask" if detected_types else "allow")  # 高敏阻断、普通 PII 掩码、无 PII 放行
    masked = normalized  # 从规范化文本开始生成安全预览
    replacements = []  # 收集实际匹配和替换标签
    for pii_type, pattern in patterns.items():  # 按类型构造掩码文本
        masked = pattern.sub(f"[{pii_type.upper()}]", masked)  # 使用类型标签替换原始值
    return {"action": action, "types": sorted(detected_types), "findings": findings, "safe_text": masked if action != "block" else "[BLOCKED_PII_OUTPUT]"}  # 返回策略证据和安全文本
focus_result = inspect_output(outputs[0]["text"])  # 检查同时含电话与邮箱的样本
print("PII-01 findings：", focus_result["findings"])  # 展示类型和 Span 中间量
print("PII-01 action：", focus_result["action"])  # 展示联系方式采用掩码策略
print("PII-01 safe_text：", focus_result["safe_text"])  # 展示可安全返回的脱敏文本


PII-01 findings： [{'type': 'email', 'start': 23, 'end': 48}, {'type': 'phone', 'start': 6, 'end': 19}]
PII-01 action： mask
PII-01 safe_text： 联系人电话 [PHONE],邮箱 [EMAIL]。


## 失败案例与修正：全角数字和零宽字符绕过

攻击者可把电话号码写成全角数字，并插入零宽空格。ASCII 正则直接扫描会漏检；NFKC 加零宽清理后恢复为可检测号码。

In [4]:
obfuscated_phone = "联系电话：１３８​００００​００００"  # 构造全角数字和零宽字符混合的绕过文本
baseline_obfuscated = baseline_detect(obfuscated_phone)  # 使用未规范化基线检查绕过文本
normalized_obfuscated = normalize_text(obfuscated_phone)  # 执行 NFKC 和零宽清理
safe_obfuscated = inspect_output(obfuscated_phone)  # 使用完整策略层重新检测
print("绕过文本：", repr(obfuscated_phone))  # 展示不可见零宽字符的原始形式
print("基线命中：", baseline_obfuscated)  # 展示 ASCII 规则漏检
print("规范化后：", normalized_obfuscated)  # 展示全角与零宽处理结果
print("修正后：", safe_obfuscated["action"], safe_obfuscated["types"], safe_obfuscated["safe_text"])  # 展示电话被识别并掩码


绕过文本： '联系电话：１３８\u200b００００\u200b００００'
基线命中： []
规范化后： 联系电话:13800000000
修正后： mask ['phone'] 联系电话:[PHONE]


## 结果表：五条输出的策略决定与泄漏率

In [5]:
policy_rows = []  # 收集五条输出的检测类型、决定和安全文本
baseline_leaks = 0  # 统计基线对高敏或应掩码样本的泄漏
core_leaks = 0  # 统计核心策略动作错误数量
print("id | expected | action | types | safe_text")  # 输出逐样本策略结果
for item in outputs:  # 对五条虚构模型输出运行独立门禁
    result = inspect_output(item["text"])  # 获取检测证据和安全动作
    policy_rows.append((item["id"], result))  # 保存结果供回归测试
    baseline_action = "mask" if baseline_detect(item["text"]) else "allow"  # 把基线检测转换为简单掩码或放行动作
    baseline_leaks += int(baseline_action != item["expected"] and item["expected"] != "allow")  # 统计需保护样本上的错误动作
    core_leaks += int(result["action"] != item["expected"])  # 统计核心策略与人工期望不一致数量
    print(f"{item['id']} | {item['expected']} | {result['action']} | {result['types']} | {result['safe_text']}")  # 展示放行、掩码和阻断结果
print(f"保护失败数：baseline={baseline_leaks}/4，core={core_leaks}/4；安全普通输出放行={policy_rows[-1][1]['action'] == 'allow'}")  # 输出保护与可用性指标


id | expected | action | types | safe_text
PII-01 | mask | mask | ['email', 'phone'] | 联系人电话 [PHONE],邮箱 [EMAIL]。
PII-02 | block | block | ['bank_card', 'national_id'] | [BLOCKED_PII_OUTPUT]
PII-03 | block | block | ['bank_card'] | [BLOCKED_PII_OUTPUT]
PII-04 | mask | mask | ['address'] | [ADDRESS],请确认。
PII-05 | allow | allow | [] | 订单 A-100 已退款,预计三个工作日到账。
保护失败数：baseline=3/4，core=0/4；安全普通输出放行=True


## 结果解读

PII-01 的电话和邮箱被分别标记并替换，用户仍可获得不含明文联系信息的回复。证件和银行卡触发整条阻断，避免部分掩码遗漏上下文。全角加零宽反例证明检测前规范化是必要步骤；PII-05 保持放行，说明门禁不能只追求拦截率。

## 生产边界

生产 PII 检测需要 NER、校验位、国家与地区规则、结构化字段递归扫描、上下文授权和误报申诉。日志不得保存被阻断原文，规则版本与策略决定需审计。正则无法可靠识别人名和自由地址，还要防 Base64、图片 OCR 和工具返回泄漏。本例号码均为虚构。

## 最小回归测试

In [6]:
assert len(outputs) >= 5  # 保证策略评测覆盖五种真实输出场景
assert focus_result["action"] == "mask" and set(focus_result["types"]) == {"email", "phone"}  # 保证电话与邮箱被同时掩码
assert baseline_obfuscated == [] and safe_obfuscated["types"] == ["phone"]  # 保证 Unicode 绕过被规范化修正
assert dict(policy_rows)["PII-02"]["action"] == "block"  # 保证虚构证件号触发整条阻断
assert dict(policy_rows)["PII-03"]["action"] == "block"  # 保证虚构银行卡号触发整条阻断
assert core_leaks == 0 and baseline_leaks > core_leaks  # 保证核心策略在五条教学样本上消除基线保护失败
